# 03 - Final Summary
## Model Comparison & Embedding Analysis
**Team:** Gradient_Issues

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from scripts.preprocess import DataLoader
from models.encoder import TrafficEncoderBiLSTM
from utils.helpers import load_results

### Load Results

In [ ]:
baselines = load_results('../results/baseline_results.json')
bilstm = load_results('../results/bilstm_results.json')
contrastive = load_results('../results/contrastive_results.json')

print("Baselines:", json.dumps(baselines, indent=2))
print("\nBiLSTM:", json.dumps(bilstm, indent=2))
print("\nContrastive:", json.dumps(contrastive, indent=2))

### Model Comparison

In [ ]:
models = ['Logistic Regression', 'Random Forest', 'BiLSTM (Supervised)', 'BiLSTM (Contrastive)']
accuracies = [
    baselines.get('LogisticRegression', {}).get('accuracy', 0),
    baselines.get('RandomForest', {}).get('accuracy', 0),
    bilstm.get('accuracy', 0),
    contrastive.get('accuracy', 0)
]

plt.figure(figsize=(10, 6))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bars = plt.bar(models, accuracies, color=colors)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.ylabel('Accuracy', fontsize=12)
plt.title('Model Comparison: Traffic Classification', fontsize=14, fontweight='bold')
plt.ylim([0.7, 1.0])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Embedding Visualization

In [ ]:
dataloader = DataLoader('../data/5g_traffic_classification.csv')
test_ds = dataloader.get_tf_dataset(*dataloader.test_data, shuffle=False)
label_names = dataloader.label_encoder.classes_

model = TrafficEncoderBiLSTM(num_classes=len(label_names), embedding_dim=32)
model.load_weights('../models/best_model_contrastive.h5')

all_embeddings, all_labels = [], []
for x_batch, y_batch in test_ds:
    _, embeddings = model(x_batch, training=False)
    all_embeddings.append(embeddings.numpy())
    all_labels.append(y_batch.numpy())

embeddings = np.vstack(all_embeddings)
labels = np.concatenate(all_labels)

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(12, 8))
colors = plt.cm.tab10(np.linspace(0, 1, len(label_names)))

for i, label_name in enumerate(label_names):
    mask = labels == i
    plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
               c=[colors[i]], label=label_name, alpha=0.6, s=30)

plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.title('Contrastive Embeddings (t-SNE)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()